In [2]:
import os

# --- INSERISCI I TUOI DATI QUI ---
os.environ['KAGGLE_USERNAME'] = "" # Scrivilo tra le virgolette
os.environ['KAGGLE_KEY'] = ""
# --------------------------------

# 1. Scarica il dataset (circa 42GB)
print("Inizio download... l'operazione richiede tempo data la dimensione del dataset.")
!kaggle datasets download -d nih-chest-xrays/data

# 2. Estrazione (opzionale - decommenta se vuoi scompattare subito)
# print("Estrazione in corso...")
# !unzip -q data.zip -d nih_data
# print("Fatto!")

Inizio download... l'operazione richiede tempo data la dimensione del dataset.
Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/data
License(s): CC0-1.0
100% 42.0G/42.0G [05:56<00:00, 126MB/s]



In [3]:
# 2. Estrazione (opzionale - decommenta se vuoi scompattare subito)
print("Estrazione in corso...")
!unzip -q data.zip -d nih_data
print("Fatto!")

Estrazione in corso...
Fatto!


In [6]:
!pwd

/content


In [7]:
import os
import pandas as pd
import shutil
from glob import glob
from sklearn.model_selection import train_test_split

# 1. PERCORSI AGGIORNATI (basati sul tuo screenshot)
DATASET_NAME = "NIH_Target_DA_Ready"
BASE_PATH = '/content/nih_data' # La cartella che vedo nello screen
CSV_PATH = os.path.join(BASE_PATH, 'Data_Entry_2017.csv')
TRAIN_VAL_TXT = os.path.join(BASE_PATH, 'train_val_list.txt')
TEST_TXT = os.path.join(BASE_PATH, 'test_list.txt')

# Il pattern cerca le immagini nelle sottocartelle images_001, images_002...
# Di solito la struttura è nih_data/images_001/images/*.png
IMAGE_GLOB_PATTERN = os.path.join(BASE_PATH, 'images_*', 'images', '*.png')

# 2. CARICAMENTO E MAPPATURA
print("Mappatura percorsi immagini in corso...")
all_image_paths = {os.path.basename(x): x for x in glob(IMAGE_GLOB_PATTERN)}

if len(all_image_paths) == 0:
    # Provo un pattern alternativo se il primo fallisce (senza la sottocartella /images/)
    IMAGE_GLOB_PATTERN = os.path.join(BASE_PATH, 'images_*', '*.png')
    all_image_paths = {os.path.basename(x): x for x in glob(IMAGE_GLOB_PATTERN)}

print(f"Trovate {len(all_image_paths)} immagini totali.")

df = pd.read_csv(CSV_PATH)
df = df[df['Patient Age'].between(19, 100)]

# 3. FILTRAGGIO E SPLIT
with open(TRAIN_VAL_TXT, 'r') as f:
    official_train_val = set(f.read().splitlines())
with open(TEST_TXT, 'r') as f:
    official_test = set(f.read().splitlines())

def get_clean_split(image_list):
    temp_df = df[df['Image Index'].isin(image_list)].copy()
    pneu = temp_df[temp_df['Finding Labels'].str.contains('Pneumonia')].copy()
    norm = temp_df[temp_df['Finding Labels'] == 'No Finding'].copy()
    return pneu, norm

pneu_train_val, norm_train_val = get_clean_split(official_train_val)
pneu_test, norm_test = get_clean_split(official_test)

pneu_train, pneu_val = train_test_split(pneu_train_val, test_size=0.1, random_state=42)
norm_train, norm_val = train_test_split(norm_train_val, test_size=0.1, random_state=42)

# Bilanciamento
norm_train = norm_train.sample(n=min(3000, len(norm_train)), random_state=42)
n_val = min(len(pneu_val), 100)
pneu_val, norm_val = pneu_val.sample(n=n_val, random_state=42), norm_val.sample(n=n_val, random_state=42)
n_test = min(len(pneu_test), 490)
pneu_test, norm_test = pneu_test.sample(n=n_test, random_state=42), norm_test.sample(n=n_test, random_state=42)

# 4. CREAZIONE FISICA
splits = {'train': (pneu_train, norm_train), 'val': (pneu_val, norm_val), 'test': (pneu_test, norm_test)}

for s, (p_df, n_df) in splits.items():
    os.makedirs(os.path.join(DATASET_NAME, s, 'pneumonia'), exist_ok=True)
    os.makedirs(os.path.join(DATASET_NAME, s, 'normal'), exist_ok=True)

    print(f"Copiando {s}...")
    for img_idx in p_df['Image Index']:
        if img_idx in all_image_paths:
            shutil.copy(all_image_paths[img_idx], os.path.join(DATASET_NAME, s, 'pneumonia', img_idx))
    for img_idx in n_df['Image Index']:
        if img_idx in all_image_paths:
            shutil.copy(all_image_paths[img_idx], os.path.join(DATASET_NAME, s, 'normal', img_idx))

# 5. COMPRESSIONE E PULIZIA (Per non esaurire i 2.9GB)
print("Compressione in corso...")
shutil.make_archive(DATASET_NAME, 'zip', DATASET_NAME)
shutil.rmtree(DATASET_NAME) # Cancella la cartella non zippata per liberare spazio

print(f"\nFATTO! Scarica {DATASET_NAME}.zip dalla colonna a sinistra.")

Mappatura percorsi immagini in corso...
Trovate 112120 immagini totali.
Copiando train...
Copiando val...
Copiando test...
Compressione in corso...

FATTO! Scarica NIH_Target_DA_Ready.zip dalla colonna a sinistra.
